In [1]:
"""
Participant-level paired statistical analysis for Reviewer 1, Comment 6.

This script does not train or reinfer any model. It reads the six consensus
prediction CSV files, verifies that they contain the same 120 participants,
and produces:

1. Exact participant-level McNemar tests: CTE-Net versus each baseline.
2. Holm-adjusted p-values across the five comparisons.
3. Paired stratified-bootstrap 95% confidence intervals for differences in
   accuracy, sensitivity, specificity, precision, F1-score, and ROC-AUC.
4. Descriptive consensus metrics for every model and test fold.
5. CSV and LaTeX tables ready for inspection and manuscript preparation.

The ten random seeds are not used as independent statistical observations.
Each CSV must contain one consensus prediction per participant.
"""

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import roc_auc_score


# =============================================================================
# 1. KAGGLE PATHS AND ANALYSIS SETTINGS
# =============================================================================

INPUT_ROOT = Path(
    "/kaggle/input/datasets/alejandragomezr/"
    "predictions-models-cte-net"
)

OUTPUT_ROOT = Path(
    "/kaggle/working/CTE_Net_participant_level_pairwise_tests"
)

THRESHOLD = 0.5
N_BOOTSTRAP = 5000
RANDOM_STATE = 20260827
EXPECTED_SUBJECTS = 120
EXPECTED_PER_CLASS = 60
EXPECTED_SEEDS = 10

MODEL_FILES = {
    "CTE-Net": "CTE_Net_subject_level_consensus_predictions.csv",
    "EEGNet": "EEGNet_subject_level_consensus_predictions.csv",
    "ShallowConvNet": (
        "ShallowConvNet_subject_level_consensus_predictions.csv"
    ),
    "T-GARNet": "TGARNet_subject_level_consensus_predictions.csv",
    "IMC-BGT": "IMCBGT_subject_level_consensus_predictions.csv",
    "MultiStream": "MultiStream_subject_level_consensus_predictions.csv",
}

BASELINE_MODELS = [
    "EEGNet",
    "ShallowConvNet",
    "T-GARNet",
    "IMC-BGT",
    "MultiStream",
]

METRICS = [
    "accuracy",
    "sensitivity",
    "specificity",
    "precision",
    "f1_score",
    "roc_auc",
]

METRIC_LABELS = {
    "accuracy": "Accuracy",
    "sensitivity": "Sensitivity",
    "specificity": "Specificity",
    "precision": "Precision",
    "f1_score": "F1-score",
    "roc_auc": "ROC-AUC",
}

COLUMN_ALIASES = {
    "subject_id": [
        "subject_id",
        "participant_id",
        "subject",
        "participant",
    ],
    "label": [
        "label",
        "y_true",
        "true_label",
        "target",
    ],
    "probability": [
        "mean_prob_adhd_across_seeds",
        "consensus_probability",
        "consensus_prob",
        "subject_prob_adhd",
        "prob_adhd",
        "y_prob",
    ],
    "prediction": [
        "consensus_pred",
        "subject_pred",
        "y_pred",
        "prediction",
    ],
    "fold": [
        "fold",
        "test_fold",
        "fold_id",
    ],
    "n_seeds": [
        "n_seeds",
        "number_of_seeds",
    ],
}


# =============================================================================
# 2. INPUT DISCOVERY, STANDARDIZATION, AND VALIDATION
# =============================================================================

def find_unique_file(root, filename):
    """Find exactly one recursively matching file under a Kaggle input root."""
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(
            f"The Kaggle input directory does not exist:\n{root}"
        )

    matches = sorted(root.rglob(filename))
    if not matches:
        available = sorted(path.name for path in root.rglob("*.csv"))
        raise FileNotFoundError(
            f"Could not find {filename} under:\n{root}\n"
            f"Available CSV files: {available}"
        )
    if len(matches) > 1:
        raise RuntimeError(
            f"Found more than one file named {filename}:\n"
            + "\n".join(str(path) for path in matches)
        )
    return matches[0]


def resolve_column(frame, canonical_name, required=True):
    """Resolve one canonical variable using the accepted column aliases."""
    lower_to_original = {
        str(column).strip().lower(): column
        for column in frame.columns
    }

    for alias in COLUMN_ALIASES[canonical_name]:
        if alias.lower() in lower_to_original:
            return lower_to_original[alias.lower()]

    if required:
        raise KeyError(
            f"Could not resolve the '{canonical_name}' column. "
            f"Available columns: {list(frame.columns)}"
        )
    return None


def clean_subject_id(value):
    """Normalize participant identifiers without changing their identity."""
    value = str(value).strip()
    if value.lower().endswith(".mat"):
        value = value[:-4]
    return value


def normalize_binary_labels(values):
    """Accept numeric labels or common Control/ADHD text labels."""
    series = pd.Series(values).copy()

    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().all():
        result = numeric.astype(int)
    else:
        normalized = series.astype(str).str.strip().str.lower()
        mapping = {
            "control": 0,
            "healthy": 0,
            "hc": 0,
            "0": 0,
            "adhd": 1,
            "tdah": 1,
            "1": 1,
        }
        result = normalized.map(mapping)
        if result.isna().any():
            invalid = sorted(normalized[result.isna()].unique())
            raise ValueError(f"Unrecognized class labels: {invalid}")
        result = result.astype(int)

    if not result.isin([0, 1]).all():
        raise ValueError("Labels must be binary: 0=Control and 1=ADHD.")
    return result.to_numpy(dtype=np.int64)


def load_consensus_predictions(model_name, csv_path):
    """Load and validate one consensus-prediction file."""
    raw = pd.read_csv(csv_path)

    subject_column = resolve_column(raw, "subject_id")
    label_column = resolve_column(raw, "label")
    probability_column = resolve_column(raw, "probability")
    prediction_column = resolve_column(raw, "prediction", required=False)
    fold_column = resolve_column(raw, "fold", required=False)
    n_seeds_column = resolve_column(raw, "n_seeds", required=False)

    standardized = pd.DataFrame(
        {
            "subject_id": raw[subject_column].map(clean_subject_id),
            "label": normalize_binary_labels(raw[label_column]),
            "probability": pd.to_numeric(
                raw[probability_column], errors="raise"
            ).astype(float),
        }
    )

    if fold_column is not None:
        standardized["fold"] = pd.to_numeric(
            raw[fold_column], errors="raise"
        ).astype(int)
    else:
        standardized["fold"] = pd.NA

    if n_seeds_column is not None:
        standardized["n_seeds"] = pd.to_numeric(
            raw[n_seeds_column], errors="raise"
        ).astype(int)
    else:
        standardized["n_seeds"] = EXPECTED_SEEDS

    if standardized["subject_id"].eq("").any():
        raise ValueError(f"{model_name}: empty participant identifier found.")
    if standardized["subject_id"].duplicated().any():
        duplicated = standardized.loc[
            standardized["subject_id"].duplicated(keep=False),
            "subject_id",
        ].tolist()
        raise RuntimeError(
            f"{model_name}: duplicated participants found: {duplicated}"
        )
    if not np.isfinite(standardized["probability"]).all():
        raise ValueError(f"{model_name}: probability contains NaN or Inf.")
    if not standardized["probability"].between(0.0, 1.0).all():
        raise ValueError(
            f"{model_name}: probabilities must lie between 0 and 1."
        )

    standardized["prediction"] = (
        standardized["probability"] >= THRESHOLD
    ).astype(np.int64)

    if prediction_column is not None:
        exported_prediction = pd.to_numeric(
            raw[prediction_column], errors="raise"
        ).astype(int).to_numpy()
        if not np.array_equal(
            exported_prediction,
            standardized["prediction"].to_numpy(),
        ):
            disagreement_count = int(
                np.sum(
                    exported_prediction
                    != standardized["prediction"].to_numpy()
                )
            )
            raise RuntimeError(
                f"{model_name}: {disagreement_count} exported consensus "
                "decisions do not match probability >= 0.5."
            )

    standardized["correct"] = (
        standardized["prediction"] == standardized["label"]
    ).astype(np.int64)
    standardized["model"] = model_name

    n_subjects = standardized["subject_id"].nunique()
    class_counts = standardized["label"].value_counts().sort_index().to_dict()
    if n_subjects != EXPECTED_SUBJECTS:
        raise RuntimeError(
            f"{model_name}: expected {EXPECTED_SUBJECTS} participants, "
            f"but found {n_subjects}."
        )
    if class_counts != {0: EXPECTED_PER_CLASS, 1: EXPECTED_PER_CLASS}:
        raise RuntimeError(
            f"{model_name}: expected 60 Control and 60 ADHD participants, "
            f"but found {class_counts}."
        )
    if not (standardized["n_seeds"] == EXPECTED_SEEDS).all():
        observed = standardized["n_seeds"].value_counts().to_dict()
        raise RuntimeError(
            f"{model_name}: every consensus prediction must use "
            f"{EXPECTED_SEEDS} seeds. Observed: {observed}"
        )

    return standardized.sort_values("subject_id").reset_index(drop=True)


def validate_paired_cohort(predictions):
    """Ensure all models contain the same participants, labels, and folds."""
    reference = predictions["CTE-Net"].set_index("subject_id").sort_index()

    for model_name, frame in predictions.items():
        current = frame.set_index("subject_id").sort_index()

        if not current.index.equals(reference.index):
            missing = sorted(set(reference.index) - set(current.index))
            extra = sorted(set(current.index) - set(reference.index))
            raise RuntimeError(
                f"{model_name}: participant set differs from CTE-Net. "
                f"Missing={missing}; extra={extra}"
            )

        if not np.array_equal(
            current["label"].to_numpy(),
            reference["label"].to_numpy(),
        ):
            raise RuntimeError(
                f"{model_name}: labels do not match CTE-Net by participant."
            )

        both_have_folds = (
            current["fold"].notna().all()
            and reference["fold"].notna().all()
        )
        if both_have_folds and not np.array_equal(
            current["fold"].to_numpy(),
            reference["fold"].to_numpy(),
        ):
            raise RuntimeError(
                f"{model_name}: test folds do not match CTE-Net by participant."
            )


# =============================================================================
# 3. METRICS, EXACT MCNEMAR TEST, AND HOLM CORRECTION
# =============================================================================

def safe_divide(numerator, denominator):
    return float(numerator / denominator) if denominator else 0.0


def calculate_metrics(y_true, probability):
    """Calculate participant-level metrics using ADHD as the positive class."""
    y_true = np.asarray(y_true, dtype=np.int64)
    probability = np.asarray(probability, dtype=np.float64)
    prediction = (probability >= THRESHOLD).astype(np.int64)

    tp = int(np.sum((y_true == 1) & (prediction == 1)))
    tn = int(np.sum((y_true == 0) & (prediction == 0)))
    fp = int(np.sum((y_true == 0) & (prediction == 1)))
    fn = int(np.sum((y_true == 1) & (prediction == 0)))

    sensitivity = safe_divide(tp, tp + fn)
    specificity = safe_divide(tn, tn + fp)
    precision = safe_divide(tp, tp + fp)
    f1_score = safe_divide(
        2.0 * precision * sensitivity,
        precision + sensitivity,
    )

    return {
        "accuracy": safe_divide(tp + tn, len(y_true)),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "f1_score": f1_score,
        "roc_auc": float(roc_auc_score(y_true, probability)),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def exact_mcnemar(correct_cte, correct_baseline):
    """Return discordant counts and the two-sided exact McNemar p-value."""
    correct_cte = np.asarray(correct_cte, dtype=bool)
    correct_baseline = np.asarray(correct_baseline, dtype=bool)

    n10 = int(np.sum(correct_cte & ~correct_baseline))
    n01 = int(np.sum(~correct_cte & correct_baseline))
    n_discordant = n10 + n01

    if n_discordant == 0:
        p_value = 1.0
    else:
        p_value = float(
            binomtest(
                k=n10,
                n=n_discordant,
                p=0.5,
                alternative="two-sided",
            ).pvalue
        )

    return n10, n01, n_discordant, p_value


def holm_adjust(p_values):
    """Holm step-down family-wise-error correction without extra packages."""
    p_values = np.asarray(p_values, dtype=np.float64)
    m = len(p_values)
    order = np.argsort(p_values)
    sorted_p = p_values[order]

    adjusted_sorted = np.empty(m, dtype=np.float64)
    running_maximum = 0.0
    for position, p_value in enumerate(sorted_p):
        candidate = (m - position) * p_value
        running_maximum = max(running_maximum, candidate)
        adjusted_sorted[position] = min(running_maximum, 1.0)

    adjusted = np.empty(m, dtype=np.float64)
    adjusted[order] = adjusted_sorted
    return adjusted


# =============================================================================
# 4. PAIRED PARTICIPANT-LEVEL BOOTSTRAP
# =============================================================================

def create_stratified_bootstrap_indices(
    labels,
    n_bootstrap=N_BOOTSTRAP,
    random_state=RANDOM_STATE,
):
    """Use the same stratified participant resamples for all model pairs."""
    labels = np.asarray(labels, dtype=np.int64)
    control_indices = np.flatnonzero(labels == 0)
    adhd_indices = np.flatnonzero(labels == 1)
    rng = np.random.default_rng(random_state)

    bootstrap_indices = np.empty(
        (n_bootstrap, len(labels)),
        dtype=np.int64,
    )

    for replicate in range(n_bootstrap):
        bootstrap_indices[replicate] = np.concatenate(
            [
                rng.choice(
                    control_indices,
                    size=len(control_indices),
                    replace=True,
                ),
                rng.choice(
                    adhd_indices,
                    size=len(adhd_indices),
                    replace=True,
                ),
            ]
        )
    return bootstrap_indices


def paired_metric_bootstrap(
    labels,
    probability_cte,
    probability_baseline,
    bootstrap_indices,
):
    """Bootstrap CTE-Net minus baseline metric differences in percentage points."""
    labels = np.asarray(labels, dtype=np.int64)
    probability_cte = np.asarray(probability_cte, dtype=np.float64)
    probability_baseline = np.asarray(
        probability_baseline,
        dtype=np.float64,
    )

    point_cte = calculate_metrics(labels, probability_cte)
    point_baseline = calculate_metrics(labels, probability_baseline)
    point_difference = {
        metric: 100.0 * (point_cte[metric] - point_baseline[metric])
        for metric in METRICS
    }

    distributions = np.empty(
        (len(bootstrap_indices), len(METRICS)),
        dtype=np.float64,
    )

    for replicate, indices in enumerate(bootstrap_indices):
        sampled_labels = labels[indices]
        sampled_cte = calculate_metrics(
            sampled_labels,
            probability_cte[indices],
        )
        sampled_baseline = calculate_metrics(
            sampled_labels,
            probability_baseline[indices],
        )
        distributions[replicate] = [
            100.0 * (sampled_cte[metric] - sampled_baseline[metric])
            for metric in METRICS
        ]

    lower = np.percentile(distributions, 2.5, axis=0)
    upper = np.percentile(distributions, 97.5, axis=0)

    summary = {}
    for metric_position, metric in enumerate(METRICS):
        summary[metric] = {
            "difference_percent_points": point_difference[metric],
            "ci_95_lower_percent_points": float(lower[metric_position]),
            "ci_95_upper_percent_points": float(upper[metric_position]),
        }

    distribution_frame = pd.DataFrame(
        distributions,
        columns=[f"delta_{metric}_percent_points" for metric in METRICS],
    )
    distribution_frame.insert(
        0,
        "bootstrap_replicate",
        np.arange(1, len(distribution_frame) + 1),
    )
    return summary, distribution_frame


# =============================================================================
# 5. DESCRIPTIVE CONSENSUS AND FOLD-LEVEL SUMMARIES
# =============================================================================

def build_consensus_performance(predictions):
    rows = []
    for model_name, frame in predictions.items():
        metrics = calculate_metrics(
            frame["label"].to_numpy(),
            frame["probability"].to_numpy(),
        )
        rows.append(
            {
                "model": model_name,
                "n_subjects": len(frame),
                **{metric: metrics[metric] for metric in METRICS},
                "tp": metrics["tp"],
                "tn": metrics["tn"],
                "fp": metrics["fp"],
                "fn": metrics["fn"],
            }
        )
    return pd.DataFrame(rows)


def build_fold_level_performance(predictions):
    """Descriptive only; participant-level tests remain the primary inference."""
    rows = []
    for model_name, frame in predictions.items():
        if frame["fold"].isna().any():
            continue

        for fold, fold_frame in frame.groupby("fold", sort=True):
            metrics = calculate_metrics(
                fold_frame["label"].to_numpy(),
                fold_frame["probability"].to_numpy(),
            )
            rows.append(
                {
                    "model": model_name,
                    "fold": int(fold),
                    "n_subjects": len(fold_frame),
                    **{metric: metrics[metric] for metric in METRICS},
                    "tp": metrics["tp"],
                    "tn": metrics["tn"],
                    "fp": metrics["fp"],
                    "fn": metrics["fn"],
                }
            )
    return pd.DataFrame(rows)


def build_pair_details(cte_frame, baseline_frame, baseline_name):
    paired = cte_frame[
        ["subject_id", "label", "fold", "probability", "prediction", "correct"]
    ].merge(
        baseline_frame[
            ["subject_id", "label", "fold", "probability", "prediction", "correct"]
        ],
        on="subject_id",
        validate="one_to_one",
        suffixes=("_cte", "_baseline"),
    )

    paired["baseline"] = baseline_name
    paired["discordance"] = np.select(
        [
            (paired["correct_cte"] == 1)
            & (paired["correct_baseline"] == 0),
            (paired["correct_cte"] == 0)
            & (paired["correct_baseline"] == 1),
        ],
        [
            "CTE_correct_baseline_incorrect",
            "CTE_incorrect_baseline_correct",
        ],
        default="concordant",
    )
    return paired


# =============================================================================
# 6. LATEX OUTPUT
# =============================================================================

def format_p_value(value):
    return f"{value:.4f}"


def write_latex_mcnemar_table(results, output_path):
    lines = [
        r"\begin{table}[H]",
        r"\centering",
        (
            r"\caption{\dc{Participant-level paired comparisons between "
            r"CTE-Net and the baseline models using consensus predictions "
            r"across ten random training seeds.}}"
        ),
        r"\label{tab:participant_mcnemar}",
        r"\scriptsize",
        r"\renewcommand{\arraystretch}{1.10}",
        r"\begin{tabular}{lccccc}",
        r"\toprule",
        (
            r"\dc{\textbf{Baseline}} & \dc{\boldmath{$n_{10}$}} & "
            r"\dc{\boldmath{$n_{01}$}} & "
            r"\dc{\shortstack{\boldmath{$\Delta$ Accuracy}\\"
            r"\textbf{(95\% CI)}}} & "
            r"\dc{\boldmath{$p_{\mathrm{exact}}$}} & "
            r"\dc{\boldmath{$p_{\mathrm{Holm}}$}} \\"
        ),
        r"\midrule",
    ]

    for _, row in results.iterrows():
        difference = (
            f"{row['delta_accuracy_percent_points']:.1f} "
            f"({row['delta_accuracy_ci_lower']:.1f}--"
            f"{row['delta_accuracy_ci_upper']:.1f})"
        )
        lines.append(
            f"\\dc{{{row['baseline']}}} & "
            f"\\dc{{{int(row['n10_cte_correct_baseline_incorrect'])}}} & "
            f"\\dc{{{int(row['n01_cte_incorrect_baseline_correct'])}}} & "
            f"\\dc{{{difference}}} & "
            f"\\dc{{{format_p_value(row['mcnemar_exact_p'])}}} & "
            f"\\dc{{{format_p_value(row['mcnemar_holm_p'])}}} \\\\"
        )

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\vspace{1mm}",
            r"\begin{minipage}{0.92\textwidth}",
            r"\footnotesize",
            r"\raggedright",
            (
                r"\dc{\textit{Note:} $n_{10}$ denotes participants correctly "
                r"classified by CTE-Net but incorrectly classified by the "
                r"corresponding baseline; $n_{01}$ denotes the reverse. "
                r"Accuracy differences are expressed in percentage points as "
                r"CTE-Net minus baseline. Exact McNemar $p$-values were adjusted "
                r"across the five comparisons using the Holm procedure.}"
            ),
            r"\end{minipage}",
            r"\end{table}",
        ]
    )
    Path(output_path).write_text("\n".join(lines) + "\n", encoding="utf-8")


# =============================================================================
# 7. COMPLETE ANALYSIS
# =============================================================================

def main():
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print("LOADING AND VALIDATING CONSENSUS PREDICTIONS")
    print("=" * 80)

    predictions = {}
    input_paths = {}
    for model_name, filename in MODEL_FILES.items():
        input_path = find_unique_file(INPUT_ROOT, filename)
        input_paths[model_name] = input_path
        predictions[model_name] = load_consensus_predictions(
            model_name,
            input_path,
        )
        print(f"{model_name:16s}: {input_path}")

    validate_paired_cohort(predictions)
    print("\nValidation passed: same 120 participants and labels in all models.")

    reference = predictions["CTE-Net"].sort_values("subject_id").reset_index(
        drop=True
    )
    labels = reference["label"].to_numpy(dtype=np.int64)
    probability_cte = reference["probability"].to_numpy(dtype=np.float64)

    bootstrap_indices = create_stratified_bootstrap_indices(
        labels,
        n_bootstrap=N_BOOTSTRAP,
        random_state=RANDOM_STATE,
    )

    mcnemar_rows = []
    difference_rows = []
    bootstrap_frames = []
    pair_detail_frames = []

    for baseline_name in BASELINE_MODELS:
        baseline = predictions[baseline_name].sort_values(
            "subject_id"
        ).reset_index(drop=True)
        probability_baseline = baseline["probability"].to_numpy(
            dtype=np.float64
        )

        n10, n01, n_discordant, exact_p = exact_mcnemar(
            reference["correct"].to_numpy(dtype=bool),
            baseline["correct"].to_numpy(dtype=bool),
        )

        difference_summary, bootstrap_distribution = paired_metric_bootstrap(
            labels,
            probability_cte,
            probability_baseline,
            bootstrap_indices,
        )

        accuracy_difference = difference_summary["accuracy"]
        mcnemar_rows.append(
            {
                "comparison": f"CTE-Net vs. {baseline_name}",
                "baseline": baseline_name,
                "n_subjects": EXPECTED_SUBJECTS,
                "n10_cte_correct_baseline_incorrect": n10,
                "n01_cte_incorrect_baseline_correct": n01,
                "n_discordant": n_discordant,
                "mcnemar_exact_p": exact_p,
                "delta_accuracy_percent_points": accuracy_difference[
                    "difference_percent_points"
                ],
                "delta_accuracy_ci_lower": accuracy_difference[
                    "ci_95_lower_percent_points"
                ],
                "delta_accuracy_ci_upper": accuracy_difference[
                    "ci_95_upper_percent_points"
                ],
            }
        )

        for metric in METRICS:
            metric_result = difference_summary[metric]
            difference_rows.append(
                {
                    "comparison": f"CTE-Net vs. {baseline_name}",
                    "baseline": baseline_name,
                    "metric": metric,
                    "metric_label": METRIC_LABELS[metric],
                    **metric_result,
                }
            )

        bootstrap_distribution.insert(1, "baseline", baseline_name)
        bootstrap_frames.append(bootstrap_distribution)
        pair_detail_frames.append(
            build_pair_details(reference, baseline, baseline_name)
        )

    mcnemar_results = pd.DataFrame(mcnemar_rows)
    mcnemar_results["mcnemar_holm_p"] = holm_adjust(
        mcnemar_results["mcnemar_exact_p"].to_numpy()
    )
    mcnemar_results["significant_holm_0_05"] = (
        mcnemar_results["mcnemar_holm_p"] < 0.05
    )
    mcnemar_results["comparison_interpretation"] = np.where(
        ~mcnemar_results["significant_holm_0_05"],
        "No significant difference after Holm correction",
        np.where(
            mcnemar_results["n10_cte_correct_baseline_incorrect"]
            > mcnemar_results["n01_cte_incorrect_baseline_correct"],
            "Significant difference favoring CTE-Net",
            "Significant difference favoring baseline",
        ),
    )

    difference_summary_frame = pd.DataFrame(difference_rows)
    bootstrap_distribution_frame = pd.concat(
        bootstrap_frames,
        ignore_index=True,
    )
    paired_subject_details = pd.concat(
        pair_detail_frames,
        ignore_index=True,
    )
    consensus_performance = build_consensus_performance(predictions)
    fold_level_performance = build_fold_level_performance(predictions)

    output_paths = {
        "mcnemar_holm": OUTPUT_ROOT / "CTE_Net_vs_baselines_McNemar_Holm.csv",
        "paired_metric_differences": (
            OUTPUT_ROOT / "CTE_Net_vs_baselines_paired_metric_differences_95CI.csv"
        ),
        "bootstrap_distributions": (
            OUTPUT_ROOT / "CTE_Net_vs_baselines_paired_bootstrap_distributions.csv"
        ),
        "paired_subject_details": (
            OUTPUT_ROOT / "CTE_Net_vs_baselines_paired_subject_details.csv"
        ),
        "consensus_performance": (
            OUTPUT_ROOT / "all_models_consensus_performance.csv"
        ),
        "fold_level_performance": (
            OUTPUT_ROOT / "all_models_fold_level_consensus_metrics.csv"
        ),
        "latex_table": OUTPUT_ROOT / "participant_level_mcnemar_holm_table.tex",
    }

    mcnemar_results.to_csv(output_paths["mcnemar_holm"], index=False)
    difference_summary_frame.to_csv(
        output_paths["paired_metric_differences"], index=False
    )
    bootstrap_distribution_frame.to_csv(
        output_paths["bootstrap_distributions"], index=False
    )
    paired_subject_details.to_csv(
        output_paths["paired_subject_details"], index=False
    )
    consensus_performance.to_csv(
        output_paths["consensus_performance"], index=False
    )
    fold_level_performance.to_csv(
        output_paths["fold_level_performance"], index=False
    )
    write_latex_mcnemar_table(
        mcnemar_results,
        output_paths["latex_table"],
    )

    display_columns = [
        "baseline",
        "n10_cte_correct_baseline_incorrect",
        "n01_cte_incorrect_baseline_correct",
        "delta_accuracy_percent_points",
        "delta_accuracy_ci_lower",
        "delta_accuracy_ci_upper",
        "mcnemar_exact_p",
        "mcnemar_holm_p",
        "comparison_interpretation",
    ]

    print("\n" + "=" * 80)
    print("EXACT MCNEMAR TESTS WITH HOLM CORRECTION")
    print("=" * 80)
    print(mcnemar_results[display_columns].to_string(index=False))

    print("\n" + "=" * 80)
    print("FILES SAVED")
    print("=" * 80)
    for description, path in output_paths.items():
        print(f"{description:28s}: {path}")

    print(
        "\nImportant: the seeds were not treated as independent clinical "
        "observations. Inference is paired at the participant level."
    )

    return {
        "mcnemar_holm": mcnemar_results,
        "paired_metric_differences": difference_summary_frame,
        "bootstrap_distributions": bootstrap_distribution_frame,
        "paired_subject_details": paired_subject_details,
        "consensus_performance": consensus_performance,
        "fold_level_performance": fold_level_performance,
        "output_paths": output_paths,
    }


if __name__ == "__main__":
    analysis_results = main()

LOADING AND VALIDATING CONSENSUS PREDICTIONS
CTE-Net         : /kaggle/input/datasets/alejandragomezr/predictions-models-cte-net/CTE_Net_subject_level_consensus_predictions.csv
EEGNet          : /kaggle/input/datasets/alejandragomezr/predictions-models-cte-net/EEGNet_subject_level_consensus_predictions.csv
ShallowConvNet  : /kaggle/input/datasets/alejandragomezr/predictions-models-cte-net/ShallowConvNet_subject_level_consensus_predictions.csv
T-GARNet        : /kaggle/input/datasets/alejandragomezr/predictions-models-cte-net/TGARNet_subject_level_consensus_predictions.csv
IMC-BGT         : /kaggle/input/datasets/alejandragomezr/predictions-models-cte-net/IMCBGT_subject_level_consensus_predictions.csv
MultiStream     : /kaggle/input/datasets/alejandragomezr/predictions-models-cte-net/MultiStream_subject_level_consensus_predictions.csv

Validation passed: same 120 participants and labels in all models.

EXACT MCNEMAR TESTS WITH HOLM CORRECTION
      baseline  n10_cte_correct_baseline_inc